# Verificación del cierre técnico — 8 de noviembre de 2026

Ejecutable sin claves ni llamadas LLM. Los controles fabricados prueban software; no son documentos originales ni validación experta. Las observaciones de evaluación se leen de un registro histórico, no se reejecutan. Instalar `requirements-test.txt` y ejecutar desde el repositorio. Los outputs fueron capturados con Python ordinario, sin kernel IPython.


In [1]:
from pathlib import Path
import json, sys
candidates = [Path.cwd(), *Path.cwd().parents, Path(__file__).resolve().parent.parent] if '__file__' in globals() else [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p/'src/risk/session_exposure_grounding.py').exists()), None)
if ROOT is None:
    raise RuntimeError('Abra este notebook desde una copia del repositorio actualizado e instale requirements-test.txt.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('Repositorio localizado. No se solicitan credenciales.')


Repositorio localizado. No se solicitan credenciales.


In [2]:
import hashlib
import pandas as pd
manifest = json.loads((ROOT/'config/risk_detail_manifest.json').read_text())
source = ROOT/'data/evaluation/risk_details/signals_published.csv'
assert hashlib.sha256(source.read_bytes()).hexdigest() == manifest['sha256']
signals = pd.read_csv(source)
assert len(signals) == manifest['rows'] == 649
assert int(signals.pird_status.eq('CALCULADO').sum()) == manifest['scored_rows'] == 339
print(json.dumps({'baseline_rows':649,'assigned_rows':manifest['assigned_rows'],'scored_rows':339,'sha256':manifest['sha256']},indent=2))


{
  "baseline_rows": 649,
  "assigned_rows": 515,
  "scored_rows": 339,
  "sha256": "5bf5e501c27d4ae5b99bf1f8062701124cdbd25c5dd4a6ee861981b2991498e5"
}


In [3]:
import runpy
from src.risk.session_exposure_grounding import review_session_analysis
example = runpy.run_path(str(ROOT/'tests/test_session_exposure_grounding.py'))['example_analysis']()
reviewed = review_session_analysis(example)
risk, request = reviewed['private']['scored_signals']
assert (risk['severity'], risk['probability']) == (3,4)
assert request['severity'] is None and request['probability'] is None and request['pird'] is None
assert request['exposure_status'] == 'INDETERMINADA'
assert request['exposure_grounding_original']['probability'] == 1
assert reviewed['summary']['signals_validated'] == 2
assert reviewed['summary']['signals_scored'] == 1
print(json.dumps({'kind':'SYNTHETIC_SOFTWARE_CONTROL','summary':reviewed['summary'],'audit':reviewed['exposure_grounding_audit']},ensure_ascii=False,indent=2))


{
  "kind": "SYNTHETIC_SOFTWARE_CONTROL",
  "summary": {
    "signals_validated": 2,
    "signals_scored": 1,
    "signals_pending": 1,
    "scoring_coverage": 0.5,
    "global_pird": 37.35,
    "global_level": "MEDIO",
    "profile_status": "PROVISIONAL"
  },
  "audit": {
    "version": "SESSION-CLOSURE-GROUNDING-1",
    "signals_flagged": 1,
    "item_ids": [
      "A1"
    ],
    "original_summary": {
      "signals_validated": 2,
      "signals_scored": 2,
      "signals_pending": 0,
      "scoring_coverage": 1.0,
      "global_pird": 30.0,
      "global_level": "MEDIO",
      "profile_status": "CALCULADO"
    },
    "policy": "Mantener denominador; no imputar ni usar etiquetas expertas; conservar propuestas originales."
  }
}


In [4]:
history=json.loads((ROOT/'results/closure_20261108/evaluation_summary.json').read_text())
assert history['reserved_cases']['3']['status'] == 'NO_RESUELTO'
assert history['reserved_cases']['5']['status'] == 'ABSTENCION_INDEBIDA'
print('Observación histórica; no es nueva ejecución LLM:')
print(json.dumps(history,ensure_ascii=False,indent=2))


Observación histórica; no es nueva ejecución LLM:
{
  "date": "2026-11-08",
  "base_commit": "f46526adeec0fedefc72a989d04eed30e7d1a24a",
  "baseline": {
    "signals": 649,
    "assigned": 515,
    "scored": 339,
    "pending": 176,
    "coverage": 0.658252427184466,
    "DataR": {
      "scored": 267,
      "assigned": 404,
      "pird": 51.5
    }
  },
  "expert_components": {
    "reviewed_cases": 7,
    "before_scored": 332,
    "after_scored": 339
  },
  "reserved_cases": {
    "1": {
      "status": "CORRECTO"
    },
    "2": {
      "status": "CORRECTO"
    },
    "3": {
      "status": "NO_RESUELTO",
      "expert_severity": 5,
      "expert_probability": 5,
      "model_severity": 3,
      "model_probability": 4
    },
    "4": {
      "status": "SIN_MONTO_INVENTADO",
      "limit": "La puerta de citas bloquea salidas sin referencia; no acredita razonamiento semántico del generador."
    },
    "5": {
      "status": "ABSTENCION_INDEBIDA"
    }
  },
  "reserved_session_before_